# Gradient Boosting Regression

In the previous notebook, we studied Random Forest Regression.

Our tuned Random Forest achieved a test R² score of approximately 0.813 on the California Housing dataset.

Random Forest builds many decision trees independently and combines their predictions.

Gradient Boosting takes a different approach.

Instead of building trees independently, Gradient Boosting builds them sequentially.

Each new tree tries to correct the errors made by the existing ensemble.

In this notebook, we will:

1. Understand the intuition behind Gradient Boosting.
2. Understand how residual errors are used.
3. Work through a small numerical example.
4. Understand the learning rate.
5. Train a Gradient Boosting model using Scikit-learn.
6. Evaluate its performance.
7. Experiment with important hyperparameters.
8. Tune the model using cross-validation.
9. Compare Gradient Boosting with our previous models.

The goal is not only to obtain a good score, but to understand why the algorithm works.

## 1. What Is Gradient Boosting?

Gradient Boosting is an ensemble learning technique that combines multiple weak learners, usually small decision trees, to create a stronger model.

The important difference from Random Forest is how the trees are built.

Random Forest:

- Builds many trees independently.
- Each tree makes its own prediction.
- The predictions are averaged.

Gradient Boosting:

- Builds one tree first.
- Looks at the errors made by the current model.
- Builds the next tree to correct those errors.
- Repeats this process.
- Adds the corrections from each new tree to the existing predictions.

Therefore, the trees in Gradient Boosting are dependent on the trees that came before them.

### The Core Idea

Suppose our model initially predicts:

Actual values:

[10, 20, 30, 40]

Initial predictions:

[15, 15, 15, 15]

The model is making errors.

We can calculate the residuals:

residual = actual - prediction

Residuals:

[-5, 5, 15, 25]

A new decision tree is trained to learn these residuals.

The new tree therefore learns where the previous model was underpredicting or overpredicting.

Its prediction is then added to the existing model.

This process is repeated many times.

The model gradually improves by correcting previous mistakes.


## 2. Mathematical Idea

Let the prediction made by the current ensemble after step m - 1 be:

F_(m-1)(x)

A new tree h_m(x) is trained to provide a correction.

The updated model becomes:

F_m(x) = F_(m-1)(x) + η h_m(x)

where:

- F_m(x) = new prediction
- F_(m-1)(x) = previous prediction
- h_m(x) = prediction from the new tree
- η = learning rate

The learning rate controls how much of the new tree's correction is added to the existing model.

Therefore, Gradient Boosting builds the final prediction gradually:

Initial prediction
        ↓
Correction from tree 1
        ↓
Correction from tree 2
        ↓
Correction from tree 3
        ↓
...
        ↓
Final prediction

### Why Is It Called "Boosting"?

Each individual decision tree is relatively weak.

Instead of relying on one large tree, Gradient Boosting combines many small trees.

Each new tree improves the existing model.

The sequence of weak learners is therefore "boosted" into a stronger learner.

In [1]:
import numpy as np

y = np.array([10, 20, 30, 40])

initial_prediction = np.mean(y)

print("Actual values:", y)
print("Initial prediction:", initial_prediction)

Actual values: [10 20 30 40]
Initial prediction: 25.0


### Why Do We Start With the Mean?

For squared-error regression, the mean of the target values is the best constant prediction.

Before building any trees, Gradient Boosting therefore starts with a simple baseline prediction.

For our example:

mean = 25

So the initial model predicts:

[25, 25, 25, 25]

In [2]:
residuals = y - initial_prediction

print("Actual values:", y)
print("Predictions:", np.full(len(y), initial_prediction))
print("Residuals:", residuals)

Actual values: [10 20 30 40]
Predictions: [25. 25. 25. 25.]
Residuals: [-15.  -5.   5.  15.]


## 3. Learning From the Errors

The residual is:

residual = actual - prediction

For our first observation:

10 - 25 = -15

For the second:

20 - 25 = -5

For the third:

30 - 25 = 5

For the fourth:

40 - 25 = 15

Therefore:

[-15, -5, 5, 15]

These residuals tell us how the current model needs to change.

Negative residual:
The model predicted too high.

Positive residual:
The model predicted too low.

The next decision tree attempts to learn these residuals.

In [4]:
learning_rate = 0.1

tree_correction = residuals

new_prediction = initial_prediction + learning_rate * tree_correction

print("Initial predictions:", np.full(len(y), initial_prediction))
print("Tree correction:", tree_correction)
print("New predictions:", new_prediction)

Initial predictions: [25. 25. 25. 25.]
Tree correction: [-15.  -5.   5.  15.]
New predictions: [23.5 24.5 25.5 26.5]


### What Just Happened?

The first model predicted:

[25, 25, 25, 25]

The new tree suggested corrections:

[-15, -5, 5, 15]

But we do not apply the entire correction.

Our learning rate is 0.1.

Therefore:

new prediction = old prediction + 0.1 × correction

This gives:

[23.5, 24.5, 25.5, 26.5]

The predictions have moved toward the actual values.

The important idea is that Gradient Boosting makes small corrections repeatedly rather than making one huge correction.